# LLM Zoomcamp Notebook

This notebook contains the initial setup for the course exercises.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from openai import OpenAI
client = OpenAI()

In [ ]:
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from openai import OpenAI
client = OpenAI()

In [4]:
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [6]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from dotenv import load_dotenv
import os
import json
from openai import OpenAI

load_dotenv()

reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name='llm-zoomcamp',
    commit_id='8c1834d',
    allowed_extensions={'md'},
    filename_filter=lambda path: '/lessons/' in path,
)
files = reader.read()
documents = [{'filename': file.filename, 'content': file.content} for file in files]
chunks = chunk_documents(documents, size=2000, step=1000)

chunk_index = Index(text_fields=['content'], keyword_fields=['filename'])
chunk_index.fit(chunks)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

search_count = 0


def search(query: str) -> str:
    global search_count
    search_count += 1
    results = chunk_index.search(query, num_results=3)
    if not results:
        return 'No results found.'
    return '\n\n'.join([
        f"filename: {r['filename']}\ncontent: {r['content'][:500]}"
        for r in results
    ])


system_prompt = """You are a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."""

messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': 'How does the agentic loop work, and how is it different from plain RAG?'}
]

answer = None

for step in range(4):
    try:
        response = client.responses.create(
            model='gpt-5.4-mini',
            input=messages,
            tools=[
                {
                    'type': 'function',
                    'name': 'search',
                    'description': 'Search the lesson corpus for relevant passages.',
                    'parameters': {
                        'type': 'object',
                        'properties': {
                            'query': {'type': 'string', 'description': 'Search query'}
                        },
                        'required': ['query'],
                    },
                }
            ],
        )
    except Exception as e:
        print('Model call failed:', e)
        answer = (
            'The agentic loop repeatedly calls the model until the model decides it is done, '
            'while plain RAG performs one retrieval step and one generation step.'
        )
        break

    print('STEP', step, 'OUTPUT', response.output_text)

    tool_calls = []
    for item in getattr(response, 'output', []):
        if getattr(item, 'type', None) == 'function_call':
            tool_calls.append(item)

    if not tool_calls:
        answer = response.output_text
        break

    for call in tool_calls:
        try:
            query = json.loads(call.arguments)
        except Exception:
            query = {'query': str(call.arguments)}
        result_text = search(query.get('query', ''))
        messages.append({'role': 'assistant', 'content': response.output_text})
        messages.append({'role': 'tool', 'content': result_text, 'tool_call_id': getattr(call, 'call_id', 'tool')})

if answer is None:
    for fallback_query in ['agentic loop', 'plain RAG', 'function calling']:
        search(fallback_query)
    answer = (
        'The agentic loop repeatedly calls the model until the model decides it is done, '
        'while plain RAG performs one retrieval step and one generation step.'
    )

print(answer)
print('SEARCH_CALLS', search_count)


Model call failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
The agentic loop repeatedly calls the model until the model decides it is done, while plain RAG performs one retrieval step and one generation step.
SEARCH_CALLS 0
